# Core Validation 007 — Resumable Confirmation v1.1

Discovery is already frozen and the winner is `interference_cut`. The original confirmation seeds `80711/80712/80713` are retired after the old monolithic runner exposed 80711 and terminated during 80712 before any per-seed checkpoint was persisted. The amended confirmation uses untouched seeds `80721/80722/80723` without changing the winner, model, data, baselines, mechanism, or gate thresholds.

Enable Internet and a GPU accelerator. Before running the main cell, add the same Kaggle Secret used by the repository's historical one-cell publishers: **`GITHUB_TOKEN`**, with Contents read/write access to `ArcheLabs/mini-cells`. Optional: add `HF_TOKEN` for higher Hugging Face rate limits.

The main cell is the only execution cell required. It fresh-clones the latest research branch, validates GitHub write access before opening a new seed, runs one seed per fresh Python/CUDA process, atomically checkpoints each seed, reports and pushes after every seed, and hydrates matching canonical checkpoints after a session restart. Re-running the same cell resumes rather than rerunning completed seeds.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/core-validation-007-functional-boundary-discovery'

# Always start from the canonical remote branch. Any completed partial seed
# checkpoints were pushed after the previous seed and will be hydrated.
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch',
    'https://github.com/ArcheLabs/mini-cells.git', str(REPO),
], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'], check=True)

# Uses the repository's proven GITHUB_TOKEN + GIT_ASKPASS publisher.
subprocess.run([
    sys.executable,
    'scripts/research/orchestrate_core_validation_007_confirmation.py',
    '--branch', BRANCH,
    '--secret-name', 'GITHUB_TOKEN',
    '--push-results',
], check=True)

decision = Path('results/core-validation-007-functional-boundary-discovery/confirmation/decision.json')
gates = Path('results/core-validation-007-functional-boundary-discovery/confirmation/gate-summary.csv')
print(decision.read_text())
print(gates.read_text() if gates.exists() else 'gate-summary.csv not written yet')
